# BTC Dynamics Analysis
**Goal:** Analyze BTC trading gateway dynamics (USD proxy dominance), gateway relevance, and macro correlation (DXY + Fed Funds).

**Inputs:**
- `data/processed/Gold and Crude_Petrol/bitcoinity_cleaned.csv`
- `data/processed/USD_Index/US_Dollar_Index_cleaned.csv`
- `data/processed/Micro Time Series/Interest Rate/interest_rate.csv`

**Outputs (auto-saved):** `data/outputs/BTC_Dynamics/`


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BTC Dynamics Analysis

Answers:
1) How is dominance of USD in BTC trading volume changing over time?
2) Which fiat gateways are gaining/losing relevance (30d / 6m / 1y)?
3) Macro correlation: DXY + interest rate vs BTC USD dominance

INPUT DATA (processed):
- data/processed/Gold and Crude_Petrol/bitcoinity_cleaned.csv
- data/processed/USD_Index/US_Dollar_Index_cleaned.csv
- data/processed/Micro Time Series/Interest Rate/interest_rate.csv
- data/processed/Micro Time Series/Bitcoin Ethereum/bitcoin_etherium.csv (optional)

OUTPUTS (saved here):
- data/outputs/BTC_Dynamics/
"""

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# ROOT DETECTION
# -----------------------------
from pathlib import Path

REPO_ROOT = Path("/Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project")
print("✅ Repo root set to:", REPO_ROOT)


print("✅ Repo root detected:", REPO_ROOT)


# -----------------------------
# PATHS
# -----------------------------
BTC_PATH = REPO_ROOT / "data" / "processed" / "Gold and Crude_Petrol" / "bitcoinity_cleaned.csv"
DXY_PATH = REPO_ROOT / "data" / "processed" / "USD_Index" / "US_Dollar_Index_cleaned.csv"
RATE_PATH = REPO_ROOT / "data" / "processed" / "Micro Time Series" / "Interest Rate" / "interest_rate.csv"

# optional context (not required for outputs)
BTC_PRICE_PATH = REPO_ROOT / "data" / "processed" / "Micro Time Series" / "Bitcoin Ethereum" / "bitcoin_etherium.csv"

OUT_DIR = REPO_ROOT / "data" / "outputs" / "BTC_Dynamics"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# CONFIG / ASSUMPTIONS
# -----------------------------
# bitcoinity_cleaned.csv contains exchange volumes (not explicit fiat pairs).
# We use major USD-heavy exchanges as "USD proxy gateways".
USD_PROXY_EXCHANGES = ["coinbase", "bitstamp"]


# -----------------------------
# LOAD DATA
# -----------------------------
def load_btc_volume():
    if not BTC_PATH.exists():
        raise SystemExit(f"Missing file: {BTC_PATH}")

    df = pd.read_csv(BTC_PATH)
    if "Time" not in df.columns:
        raise SystemExit("bitcoinity_cleaned.csv must contain a 'Time' column.")

    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"]).sort_values("Time")

    exch_cols = [c for c in df.columns if c != "Time"]

    for c in exch_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["total_volume"] = df[exch_cols].sum(axis=1)

    valid_usd_cols = [c for c in USD_PROXY_EXCHANGES if c in df.columns]
    if not valid_usd_cols:
        raise SystemExit(
            f"USD proxy exchanges not found. Expected any of: {USD_PROXY_EXCHANGES}. "
            f"Columns found: {exch_cols}"
        )

    df["usd_proxy_volume"] = df[valid_usd_cols].sum(axis=1)
    df["usd_proxy_share"] = (df["usd_proxy_volume"] / df["total_volume"]) * 100

    return df, exch_cols


def load_dxy():
    if not DXY_PATH.exists():
        raise SystemExit(f"Missing file: {DXY_PATH}")

    dxy = pd.read_csv(DXY_PATH)
    if "Date" not in dxy.columns or "Price" not in dxy.columns:
        raise SystemExit("US_Dollar_Index_cleaned.csv must contain 'Date' and 'Price' columns.")

    dxy["Date"] = pd.to_datetime(dxy["Date"], errors="coerce")
    dxy = dxy.dropna(subset=["Date"]).sort_values("Date")
    dxy["Price"] = pd.to_numeric(dxy["Price"], errors="coerce")

    return dxy[["Date", "Price"]].rename(columns={"Price": "DXY"})


def load_rates():
    if not RATE_PATH.exists():
        raise SystemExit(f"Missing file: {RATE_PATH}")

    r = pd.read_csv(RATE_PATH)
    if "observation_date" not in r.columns or "FEDFUNDS" not in r.columns:
        raise SystemExit("interest_rate.csv must contain 'observation_date' and 'FEDFUNDS' columns.")

    r["observation_date"] = pd.to_datetime(r["observation_date"], errors="coerce")
    r = r.dropna(subset=["observation_date"]).sort_values("observation_date")
    r["FEDFUNDS"] = pd.to_numeric(r["FEDFUNDS"], errors="coerce")

    return r.rename(columns={"observation_date": "Date"})


# -----------------------------
# ANALYSIS FUNCTIONS
# -----------------------------
def compute_gateway_shares(df, exch_cols):
    long = df.melt(
        id_vars=["Time", "total_volume"],
        value_vars=exch_cols,
        var_name="gateway",
        value_name="volume",
    )
    long["share_pct"] = (long["volume"] / long["total_volume"]) * 100
    return long


def window_summary(long_df, days):
    end_date = long_df["Time"].max()
    start_date = end_date - pd.Timedelta(days=days)
    sub = long_df[(long_df["Time"] >= start_date) & (long_df["Time"] <= end_date)]

    summary = (
        sub.groupby("gateway")["share_pct"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    summary["window_days"] = days
    return summary


def correlation_analysis(btc_df, dxy_df, rate_df):
    # BTC weekly -> monthly mean (month start)
    btc_month = (
        btc_df.set_index("Time")[["usd_proxy_share"]]
        .resample("MS")
        .mean()
        .reset_index()
        .rename(columns={"Time": "Date"})
    )

    merged = pd.merge(btc_month, dxy_df, on="Date", how="left")
    merged = pd.merge(merged, rate_df, on="Date", how="left")

    corr = merged[["usd_proxy_share", "DXY", "FEDFUNDS"]].corr()
    return merged, corr


# -----------------------------
# PLOTS
# -----------------------------
def save_plot_usd_dominance(btc_df):
    plt.figure()
    plt.plot(btc_df["Time"], btc_df["usd_proxy_share"])
    plt.title("USD Proxy Dominance in BTC Volume (Coinbase + Bitstamp Share %)")
    plt.xlabel("Time")
    plt.ylabel("USD Proxy Share (%)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "usd_proxy_dominance_trend.png", dpi=200)
    plt.close()


def save_plot_gateway_share(long_df):
    top5 = (
        long_df.groupby("gateway")["share_pct"]
        .mean()
        .sort_values(ascending=False)
        .head(5)
        .index.tolist()
    )
    sub = long_df[long_df["gateway"].isin(top5)]

    plt.figure()
    for g in top5:
        tmp = sub[sub["gateway"] == g]
        plt.plot(tmp["Time"], tmp["share_pct"], label=g)

    plt.title("Top 5 BTC Trading Gateways Share Over Time")
    plt.xlabel("Time")
    plt.ylabel("Share (%)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "top5_gateways_share_trend.png", dpi=200)
    plt.close()


def save_plot_macro(merged_df):
    plt.figure()
    plt.plot(merged_df["Date"], merged_df["usd_proxy_share"], label="USD Proxy Share")
    plt.plot(merged_df["Date"], merged_df["DXY"], label="DXY")
    plt.title("USD Proxy BTC Dominance vs US Dollar Index (DXY)")
    plt.xlabel("Date")
    plt.tight_layout()
    plt.legend()
    plt.savefig(OUT_DIR / "usd_share_vs_dxy.png", dpi=200)
    plt.close()


# -----------------------------
# MAIN
# -----------------------------
def main():
    btc_df, exch_cols = load_btc_volume()
    dxy_df = load_dxy()
    rate_df = load_rates()

    long_df = compute_gateway_shares(btc_df, exch_cols)

    # Q1 outputs
    btc_df.to_csv(OUT_DIR / "btc_usd_proxy_dominance_timeseries.csv", index=False)

    # Q2 outputs
    summary_30d = window_summary(long_df, 30)
    summary_6m = window_summary(long_df, 180)
    summary_1y = window_summary(long_df, 365)

    gateway_summary = pd.concat([summary_30d, summary_6m, summary_1y], ignore_index=True)
    gateway_summary.to_csv(OUT_DIR / "gateway_share_windows_summary.csv", index=False)

    # Q3 outputs
    merged_macro, corr_tbl = correlation_analysis(btc_df, dxy_df, rate_df)
    merged_macro.to_csv(OUT_DIR / "btc_macro_merged_monthly.csv", index=False)
    corr_tbl.to_csv(OUT_DIR / "btc_macro_correlation_matrix.csv")

    # plots
    save_plot_usd_dominance(btc_df)
    save_plot_gateway_share(long_df)
    save_plot_macro(merged_macro)

    # Print results
    print("\n==============================")
    print("BTC Dynamics - Results")
    print("==============================\n")

    print("Q1) USD proxy dominance trend saved to:")
    print(f"   - {OUT_DIR / 'btc_usd_proxy_dominance_timeseries.csv'}")
    print(f"   - {OUT_DIR / 'usd_proxy_dominance_trend.png'}\n")

    print("Q2) Gateway relevance (30d/6m/1y) summary saved to:")
    print(f"   - {OUT_DIR / 'gateway_share_windows_summary.csv'}\n")

    print("Q3) Macro correlation saved to:")
    print(f"   - {OUT_DIR / 'btc_macro_merged_monthly.csv'}")
    print(f"   - {OUT_DIR / 'btc_macro_correlation_matrix.csv'}")
    print(f"   - {OUT_DIR / 'usd_share_vs_dxy.png'}\n")

    print("✅ All outputs saved in:", OUT_DIR)
    print("")


if __name__ == "__main__":
    main()


✅ Repo root set to: /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project
✅ Repo root detected: /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project

BTC Dynamics - Results

Q1) USD proxy dominance trend saved to:
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/BTC_Dynamics/btc_usd_proxy_dominance_timeseries.csv
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/BTC_Dynamics/usd_proxy_dominance_trend.png

Q2) Gateway relevance (30d/6m/1y) summary saved to:
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/BTC_Dynamics/gateway_share_windows_summary.csv

Q3) Macro correlation saved to:
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/BTC_Dynamics/btc_macro_merged_monthly.csv
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/BTC_Dynamics/btc_macro_correlation_matrix.csv
   - /Users/santoshd/Desktop/UE/DSB/Data-Science-Final-Project/data/outputs/BTC_D